# SmartGrade — تشغيل مباشر على Google Colab

لا تحتاج تنزيل المشروع أو رفع ZIP. اختر **Python 3 / CPU** ثم **Runtime → Run all**.
تجلب الخلية دفتر المؤسسات من نسخة ثابتة في GitHub، وتتحقق من بصمته ثم تشغّل خطواته نفسها.

أثناء التشغيل سيُطلب منك رمز المؤسسة واسمها وبريد المدير واسمه وكلمة مرور تختارها بنفسك. بعد اكتمال التثبيت تظهر شاشة الدخول داخل الدفتر. يبدأ المشروع دون اختبارات أو طلاب جاهزين؛ يمكنك إنشاء اختبارك ورفع أوراقك ومراجعة الدرجات.

**هذه هي النسخة المنشورة في المستودع عند التحديث 73b9f73. لا يشمل هذا الدفتر تعديلات الأسئلة المتعددة أو Flutter غير المرفوعة.** التصحيح المقالي يحتاج اعتماد المعلم، ولم تُقَس جودة التقييم الآلي على إجابات طلاب حقيقية.

البيانات على قرص Colab مؤقتة. احتفظ بنسخة مشفرة ومفتاحها منفصلًا باستخدام الخطوة التالية قبل إنهاء الجلسة. حفظ الدفتر وحده لا يحفظ بياناتك.

للاستعادة في جلسة جديدة: ارفع ملف .sgb والمفتاح من لوحة ملفات Colab، ثم فعّل RESTORE_PREVIOUS وحدد مساريهما قبل تشغيل الخلية. وجهة الاستعادة يجب أن تكون جديدة؛ لا تُستبدل بيانات حالية.


In [ ]:
# @title تشغيل SmartGrade دون تنزيل ملفات المشروع
"""Open the published institutional SmartGrade notebook directly in Google Colab."""

import hashlib
import json
import sys
from urllib.request import urlopen

# اترك الاستعادة معطلة في أول جلسة. لاستعادة بياناتك، ارفع ملفاتك من لوحة Colab أولًا.
RESTORE_PREVIOUS = False  # @param {type:"boolean"}
RESTORE_FILE = ""  # @param {type:"string"}
RESTORE_KEY = ""  # @param {type:"string"}

SOURCE_COMMIT = "73b9f73dc6d247f04f3b4da67fec32a0f5e20b38"
NOTEBOOK_BLOB = "3822328058e3115723a1a26921b403e969428a78"
NOTEBOOK_URL = (
    "https://raw.githubusercontent.com/alkhwarzmy/SmartGrade-AI/"
    + SOURCE_COMMIT
    + "/notebooks/SmartGrade_Institution_Colab.ipynb"
)
MAX_NOTEBOOK_BYTES = 8 * 1024 * 1024


def _source(cell):
    value = cell["source"]
    return "".join(value) if isinstance(value, list) else value


def _fetch_notebook():
    with urlopen(NOTEBOOK_URL, timeout=60) as response:
        payload = response.read(MAX_NOTEBOOK_BYTES + 1)
    if len(payload) > MAX_NOTEBOOK_BYTES:
        raise ValueError("حجم دفتر المصدر غير متوقع")
    header = b"blob " + str(len(payload)).encode("ascii") + b"\0"
    if hashlib.sha1(header + payload).hexdigest() != NOTEBOOK_BLOB:
        raise ValueError("محتوى الدفتر لا يطابق النسخة المثبتة")
    notebook = json.loads(payload)
    if notebook.get("nbformat") != 4:
        raise ValueError("صيغة دفتر المصدر غير متوقعة")
    cells = [cell for cell in notebook["cells"] if cell["cell_type"] == "code"]
    if len(cells) != 10:
        raise ValueError("عدد خلايا دفتر المصدر غير متوقع")
    for index, cell in enumerate(cells, 1):
        compile(_source(cell), f"<institution-source-{index}>", "exec")
    return cells


def start_project():
    if sys.version_info < (3, 11):
        raise RuntimeError("اختر جلسة Colab حديثة تستخدم Python 3.11 أو أحدث")
    try:
        __import__("google.colab")
    except ImportError as exc:
        raise RuntimeError("شغّل هذا الدفتر داخل Google Colab") from exc
    if RESTORE_PREVIOUS and (not RESTORE_FILE.strip() or not RESTORE_KEY.strip()):
        raise ValueError("حدد مسار النسخة المشفرة ومفتاحها قبل تفعيل الاستعادة")
    scope = globals().setdefault("_smartgrade_direct_scope", {})
    previous = scope.get("_smartgrade_institution_runtime")
    if previous is not None:
        previous.stop()
    cells = _fetch_notebook()
    globals()["_smartgrade_direct_cells"] = cells
    scope.setdefault("__name__", "__main__")
    for index, cell in enumerate(cells, 1):
        source = _source(cell)
        if "RESTORE_INSTITUTION = False" in source:
            source = source.replace(
                "RESTORE_INSTITUTION = False",
                "RESTORE_INSTITUTION = " + repr(bool(RESTORE_PREVIOUS)),
                1,
            )
            source = source.replace('RESTORE_ARCHIVE = ""', "RESTORE_ARCHIVE = " + repr(RESTORE_FILE), 1)
            source = source.replace('RESTORE_KEY_FILE = ""', "RESTORE_KEY_FILE = " + repr(RESTORE_KEY), 1)
        print(f"تشغيل خطوة {index} من {len(cells)}…", flush=True)
        exec(compile(source, f"<institution-step-{index}>", "exec"), scope)
    print("استخدم شاشة الدخول التي ظهرت أعلاه. احفظ نسخة مشفرة قبل انتهاء جلسة Colab.")


def _run_original(marker, replacements=None):
    cells = globals().get("_smartgrade_direct_cells")
    scope = globals().get("_smartgrade_direct_scope")
    if not cells or scope is None:
        raise RuntimeError("شغّل خلية بدء المشروع أولًا")
    matching = [_source(cell) for cell in cells if marker in _source(cell)]
    if len(matching) != 1:
        raise RuntimeError("تعذر تحديد خطوة المصدر المطلوبة")
    source = matching[0]
    for before, after in (replacements or {}).items():
        if source.count(before) != 1:
            raise RuntimeError("شكل خطوة المصدر غير متوقع")
        source = source.replace(before, after, 1)
    exec(compile(source, "<institution-command>", "exec"), scope)


def backup_project():
    _run_original(
        'BACKUP_ACTION = "none"',
        {
            'BACKUP_ACTION = "none"': 'BACKUP_ACTION = "download"',
            "DOWNLOAD_BACKUP_KEY = False": "DOWNLOAD_BACKUP_KEY = True",
        },
    )


def reopen_project():
    _run_original("output.serve_kernel_port_as_iframe(")


def stop_project():
    _run_original("STOP_APP = False", {"STOP_APP = False": "STOP_APP = True"})


if __name__ == "__main__":
    start_project()


## حفظ العمل
فعّل SAVE_BACKUP وشغّل الخلية التالية لتنزيل البيانات المشفرة ومفتاحها. تتوقف الواجهة والعامل أولًا لضمان النسخ المتسق. احتفظ بالملفين منفصلين. لإعادة فتح الواجهة استخدم الخلية التي بعدها.

In [ ]:
# @title حفظ نسخة احتياطية ومفتاحها
SAVE_BACKUP = False  # @param {type:"boolean"}
if SAVE_BACKUP:
    backup_project()


In [ ]:
# @title إعادة فتح الواجهة بعد النسخ
REOPEN_INTERFACE = False  # @param {type:"boolean"}
if REOPEN_INTERFACE:
    reopen_project()


In [ ]:
# @title إيقاف اختياري
STOP_INTERFACE = False  # @param {type:"boolean"}
if STOP_INTERFACE:
    stop_project()


## متابعة التشغيل
- عند تعذر ظهور الواجهة، تحقق من اتصال جلسة Colab ومن السماح بإظهار محتوى الدفتر.
- OCR والتقارير الخلفية يحتاجان استمرار جلسة Colab والعامل. يمكنك تدقيق النص والتصحيح يدويًا.
- لا ينفذ هذا المشغّل طلبات LLM مدفوعة تلقائيًا.
- لم تُشغّل جلسة Google حية أثناء إعداد هذا المشغّل. يعتمد على دفتر المؤسسات المنشور وخطواته الموثقة في المستودع.
- دليل المؤسسات: https://github.com/alkhwarzmy/SmartGrade-AI/blob/main/docs/institution_colab.md
